# Notebook 06 · 韧性度量：p_c / 损失面积 / RHI / 恢复过程

**曲线要压成数字才能进论文表格**。这一节把 L0–L3 的 S(f) 曲线换算成指标：

| 指标 | 含义 | 口径出处 |
|---|---|---|
| `p_c` | 渗流阈值（崩到哪算垮） | 中位数法 / 最大跌幅法两种，报的时候写清 |
| `loss_area` | 韧性损失面积（AUC 型） | **Ouyang et al., Structural Safety 25:33 (2012)** |
| `rhi` | 混合韧性指数 = 1 − 损失面积/区间 | 档 B（预测 RHI）的预测目标 |
| `recovery_slope/time` | 恢复速率与恢复时间 | Hosseini et al., RESS 145:47 (2016) |
| `vulnerability_probe` | 单点后果 I_i = p_c⁰ − p_cⁱ | 本项目对 FINDER 资产的**脆弱性探针**定位 |

⚠️ 归因提醒：韧性损失面积的出处是 **Ouyang 2012 Structural Safety**，
不是常被混引的 Ouyang & Wang 2015 RESS（那篇讲相依系统联合恢复）。

## 环境准备（每个 notebook 都要先跑这两格）

In [ ]:
"""环境准备：定位项目根目录、加 src 到搜索路径、设好画图参数。"""
import sys
from pathlib import Path

# 从当前工作目录往上找，直到看见 src/phase6_interdependent（这样在 notebooks/ 里也能跑）
root = Path.cwd()
while not (root / "src" / "phase6_interdependent").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

RESULTS = root / "results"
RESULTS.mkdir(exist_ok=True)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("项目根目录：", root)

In [ ]:
# 公共底座：一对 BA 双层网络 + 骨架类（后面所有 notebook 都从这一格开始）
from phase6_interdependent.core import InterdependentNetwork

N, m = 300, 3
G_A = nx.barabasi_albert_graph(N, m, seed=42)   # 层 A：比如电力网
G_B = nx.barabasi_albert_graph(N, m, seed=43)   # 层 B：比如通信网
nodes_A = sorted(G_A.nodes())
print(f"双层网络就绪：N={N}, 每层边数={G_A.number_of_edges()}")

## 第 1 步

In [ ]:
from metrics.resilience_metrics import (
    percolation_threshold, transition_order, loss_area, rhi,
    recovery_metrics, vulnerability_probe
)
from phase6_interdependent.L0_static_dependency import percolation_curve

fractions = np.linspace(0.0, 0.5, 26)
_, S = percolation_curve(G_A, G_B, fractions, coupled_fraction=1.0, n_trials=3, seed=42)
fractions = np.array(fractions)

order, jump = transition_order(fractions, S)
print(f"相变阶数：{order}（最大单步跌幅 {jump:.3f}）")
print(f"p_c（中位数法）：{percolation_threshold(fractions, S, method='midpoint'):.4f}")
print(f"p_c（最大跌幅法）：{percolation_threshold(fractions, S, method='max_drop'):.4f}")
print(f"韧性损失面积：{loss_area(fractions, S):.4f}")
print(f"RHI：{rhi(fractions, S):.4f}   （越大越韧性）")

## 第 2 步

In [ ]:
# 恢复过程：性能先掉下去，再按恢复策略爬回来
times = np.array([0, 1, 2, 3, 4, 5, 6])
performance = np.array([1.0, 0.4, 0.25, 0.3, 0.6, 0.9, 1.0])
m = recovery_metrics(times, performance)
print(f"最低性能：{m['min_performance']:.2f}")
print(f"恢复斜率：{m['recovery_slope']:.2f} / 单位时间")
print(f"恢复时间：{m['recovery_time']} （回到 1.0 的时刻）")

plt.figure(figsize=(7, 4))
plt.plot(times, performance, "o-")
plt.axhline(1.0, color="gray", ls="--", lw=1)
plt.xlabel("时间"); plt.ylabel("性能水平")
plt.title("恢复过程与韧性损失面积（灰线之间就是损失）")
plt.grid(alpha=0.3)
plt.savefig(RESULTS / "notebook_06_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

## 第 3 步

In [ ]:
# 脆弱性探针：移掉一个 hub 后 p_c 变了多少（I_i = p_c⁰ − p_cⁱ）
hub = max(G_A.degree(), key=lambda kv: kv[1])[0]
G_A_minus = G_A.copy(); G_A_minus.remove_node(hub)
G_B_minus = G_B.copy(); G_B_minus.remove_node(sorted(G_B.nodes())[-1])

pc0 = percolation_threshold(fractions, S, method="midpoint")
_, S_after = percolation_curve(G_A_minus, G_B_minus, fractions,
                               coupled_fraction=1.0, n_trials=3, seed=42)
pc1 = percolation_threshold(fractions, S_after, method="midpoint")
print(f"原始网络 p_c = {pc0:.4f}，移除 hub({hub}) 后 p_c = {pc1:.4f}")
print(f"单点级联后果 I_i = {vulnerability_probe(pc0, pc1):+.4f}")

**思考题**

1. 中位数法与最大跌幅法给出的 p_c 什么时候差得最多？（一阶相变时）
2. RHI 与"韧性损失面积"是同一个信息的两种说法 —— 为什么不能当两个独立创新点报？
3. 脆弱性探针为什么只用于"定位加固对象"，而不能当成攻击策略写进论文？